<a href="https://colab.research.google.com/github/seoking1026-ai/-/blob/main/4%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# [문제9] air_quality.csv 활용문제
import pandas as pd
# df = pd.read_csv("air_quality.csv")
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/air_quality.csv")

# 1) IQR 계산
Q1 = df['CO2'].quantile(0.25)
Q3 = df['CO2'].quantile(0.75)
IQR = Q3 - Q1

# 2) 상한 및 하한 계산
upper = Q3 + 1.5 * IQR
lower = Q1 - 1.5 * IQR

# 3) 이상치 식별
outliers = df[(df['CO2'] < lower) | (df['CO2'] > upper)]

# 4) 이상치 수
print(len(outliers))

304


In [1]:
# [문제10] 대륙별 맥주 소비량 분석
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/drinks.csv")

continent = df.groupby("continent")['beer_servings'].mean()
top_continent = continent.idxmax()

cond = df['continent'] == top_continent
df = df[cond]
df = df.sort_values('beer_servings', ascending=False)
print(df.iloc[4, 1])

313


In [2]:
# [문제11] 방문객 합계 및 관광객 비율
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/tourist.csv")

df['방문객합계'] = df['관광'] + df['공무'] + df['사업'] + df['기타']
df['관광객비율'] = df['관광'] / df['방문객합계']

a = df.nlargest(2, '관광객비율').iloc[1]['사업']
b = df.nlargest(2, '관광').iloc[1]['공무']
print(a+b)

441


In [3]:
# [문제12] Min-Max 스케일링 및 표준편차
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/chem.csv")

scaler = MinMaxScaler()
df['co_scaled'] = scaler.fit_transform(df[['co']])
df['nmhc_scaled'] = scaler.fit_transform(df[['nmhc']])

co_std = df['co_scaled'].std()
nmhc_std = df['nmhc_scaled'].std()
std_diff = round(co_std - nmhc_std, 3)
print(std_diff)

-0.017


In [4]:
# [문제13] 총대출액 및 지역코드 분석
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/loan.csv")

df['총대출액'] = df['신용대출'] + df['담보대출']
grouped = df.groupby(['지역코드', '성별'])['총대출액'].sum().unstack()
grouped['차이'] = abs(grouped[1] - grouped[2])
result = grouped['차이'].idxmax()
print(result)

4100000278


In [10]:
# [문제14]
import pandas as pd
# df = pd.read_csv("crime.csv")
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/crime.csv")

# 1) "발생건수"와 "검거건수"를 따로 분리
cond1 = df["구분"] == "발생건수"
cond2 = df["구분"] == "검거건수"
df1 = df[cond1].iloc[:, 2:] # 발생건수만 가져오기
df2 = df[cond2].iloc[:, 2:] # 검거건수만 가져오기

# 2) 검거율 계산 (검거건수 / 발생건수)
df1 = df1.reset_index(drop=True)
df2 = df2.reset_index(drop=True)
df3 = df2 / df1
#reset_index(drop=True)는 필터링 후 남아 있는 행들의 인덱스를 0, 1, 2, ...처럼 다시 새로 매기는 작업. 그리고 drop=True는 기존 인덱스를 새로운 열로 남기지 않고 버린다.
# 3) 각 연도에서 검거율이 가장 높은 범죄유형 찾기
listbox = df3.idxmax(axis=1)
#df3의 각 행에서 가장 큰 값의 열 이름을 구합니다.

# 4) 가장 높은 검거율을 기록한 범죄유형의 검거건수 가져오기
result = 0
for index, item in enumerate(listbox):
     result = result + df2.loc[index, item]
print(result)

7799


In [15]:
# [문제15]
import pandas as pd
# df = pd.read_csv("hr.csv")
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/hr.csv")
# 1) 만족도 결측치 처리
m = df["만족도"].mean()
df["만족도"] = df["만족도"].fillna(m)
# 2) 근속연수 결측치 처리 (부서와 성과등급 기준 평균값으로 채움)
gm = df.groupby(["부서", "성과등급"])["근속연수"].transform("mean")
#근속연수는 전체 평균보다 같은 부서, 같은 성과등급 사람들의 평균으로 채우는 것이 더 자연스럽다

#부서와 성과등급이 같은 사람들끼리 묶어서 근속연수의 평균을 구하고, 그 평균값을 원래 각 행마다 대응되게 다시 채운 Series를 만든다
gm = gm.astype(int)
df["근속연수"] = df["근속연수"].fillna(gm)

# 3) 연봉 / 근속연수 계산 후 세 번째로 높은 사람의 근속연수 (A)
df["연봉_근속연수"] = df["연봉"] / df["근속연수"]
# “근속연수 1년당 연봉이 얼마나 되는가” 같은 비율값 생성 예) 연봉 = 6000, 근속연수 = 3 이면, "연봉_근속연수" 값은 2000

df_year = df.nlargest(3,'연봉_근속연수')
#  "연봉_근속연수" 열을 기준으로 값이 큰 순서대로 상위 3개 행을 뽑는다.

A = df_year.iloc[-1]["근속연수"]
# 의 -1은 마지막 행을 뜻합니다. 그런데 df_year는 이미 상위 3개만 뽑아 놓은 상태이므로, 그 마지막 행은 곧 상위 3명 중 3번째 사람, 즉 세 번째로 높은 사람

# 4) 연봉 / 만족도 계산 후 두 번째로 높은 사람의 교육참가횟수 (B)
df["연봉_만족도"] = df["연봉"] / df["만족도"]
df_like = df.nlargest(2,'연봉_만족도')

# "연봉_만족도" 열을 기준으로 값이 큰 순서대로 상위 2개 행을 뽑는다.
B = df_like.iloc[-1]["교육참가횟수"]
# 5) 결과 출력
result = A + B
print(result)

7.0
